# Phase 3: Exploratory Data Analysis (EDA)

**Project:** Indian E-Commerce Recommendation System  
**Dataset:** Cleaned 100k Amazon Products (`data/processed/products_cleaned.csv`)

### Objectives:
1. **Category Volume Analysis**: Examine catalog coverage across all 9 retail departments.
2. **Price Distribution in ₹ INR**: Quantify pricing characteristics across categories.
3. **Customer Review Patterns**: Assess product engagement and review volumes.
4. **Product Title Length & Vocabulary**: Prepare for TF-IDF text vectorization by analyzing word counts and lexical density.
5. **Brand Distribution**: Inspect market concentration of leading brand tokens.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(".."))
from src.data.eda_analysis import generate_eda_figures

PROCESSED_CSV = os.path.join("..", "data", "processed", "products_cleaned.csv")
FIGURES_DIR = os.path.join("..", "reports", "figures")
EDA_JSON = os.path.join("..", "reports", "eda_summary.json")

df = pd.read_csv(PROCESSED_CSV, low_memory=False)
print(f"Loaded {len(df):,} cleaned products across {df['category'].nunique()} categories.")
df.head()

## 1. Product Distribution by Category
Fashion and footwear represent the largest volume, followed by automotive, toys, and cameras.

In [ ]:
plt.figure(figsize=(10, 5))
cat_counts = df["category"].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index, palette="mako", legend=False)
plt.title("Product Catalog Distribution by Category", fontsize=13, fontweight="bold")
plt.xlabel("Product Count")
plt.ylabel("Category")
for i, v in enumerate(cat_counts.values):
    plt.text(v + 100, i, f"{v:,}", va="center")
plt.show()

## 2. Price Distribution (₹ INR) across Categories
Prices exhibit logical real-world stratification: Laptops have the highest median price (₹81,257), whereas digital media & movies have the lowest (₹747).

In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=df, x="price_inr", y="category", hue="category", palette="viridis", legend=False, showfliers=False)
plt.title("Product Price Distribution (₹ INR) across Categories (Interquartile Range)", fontsize=13, fontweight="bold")
plt.xlabel("Price in INR (₹)")
plt.ylabel("Category")
plt.show()

## 3. Product Title Length Analysis
Product titles average 11.5 words (median 9 words). This provides sufficient lexical density for unigram and bigram TF-IDF vectorization in Phase 4.

In [ ]:
title_words = df["title"].apply(lambda t: len(str(t).split()))
plt.figure(figsize=(9, 4))
sns.histplot(title_words, bins=35, kde=True, color="#1b8a5a")
plt.axvline(title_words.median(), color="red", linestyle="--", label=f"Median: {title_words.median():.0f} words")
plt.title("Product Title Word Count Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Words per Title")
plt.ylabel("Product Count")
plt.legend()
plt.show()

## 4. Key Takeaways for Feature Engineering (Phase 4)
1. **Lexical Richness**: Product titles contain clear brand tokens (`Samsung`, `Apple`, `Sony`, `Nike`, `Under Armour`) and technical specs (`128GB`, `Full HD`, `Wireless`, `Cotton`).
2. **Category Anchoring**: Prepending the normalized category to the text feature will prevent cross-category drift in recommendations (e.g., preventing a laptop recommendation when viewing men's shoes).
3. **Price Context**: Price and review signals can be used for filtering and sorting in the API.